In [ ]:
# =====================================================
# 0. 라이브러리 / 파일 설정
# =====================================================

import re
import json
import time
import pandas as pd
from tqdm import tqdm

from openai import OpenAI
client = OpenAI()

INPUT_FILE = "tottenham_reanalyze_target.xlsx"

OUTPUT_XLSX = "tottenham_reanalyze_target_labeled.xlsx"
OUTPUT_CSV = "tottenham_reanalyze_target_labeled.csv"


# =====================================================
# 1. GPT 응답 JSON 파싱 함수
# =====================================================

def parse_json_array(text):
    text = text.strip()

    # 코드블록 제거
    text = re.sub(r"^```json|^```|```$", "", text, flags=re.MULTILINE).strip()

    # JSON 배열 부분만 추출
    start = text.find("[")
    end = text.rfind("]")

    if start != -1 and end != -1:
        text = text[start:end+1]

    return json.loads(text)


# =====================================================
# 2. 데이터 불러오기
# =====================================================

df = pd.read_excel(INPUT_FILE)

df = df.reset_index(drop=True).copy()
df["analysis_id"] = df.index

df["text_clean"] = df["text_clean"].fillna("").astype(str)
df = df[df["text_clean"].str.len() >= 2].reset_index(drop=True)

print("분석 대상 데이터 수:", len(df))
print("\n[source 분포]")
print(df["source"].value_counts(dropna=False))

print("\n[content_type 분포]")
print(df["content_type"].value_counts(dropna=False))

print("\n[gpt_sentiment 기존 분포]")
print(df["gpt_sentiment"].value_counts(dropna=False))

In [ ]:
# =====================================================
# 3. match_context 생성용 라운드별 경기 맥락
# =====================================================
# 여기에 토트넘 29R~38R 경기 결과/맥락을 직접 입력

match_context_map = {
    "29R": "프리미어리그 29라운드(vs 크리스탈 팰리스): 토트넘 1:3 패배. 토트넘과 상대팀 모두 득점한 경기.",
    "30R": "프리미어리그 30라운드(vs 리버풀): 토트넘 1:1 무승부. 토트넘과 상대팀 모두 득점한 경기.",
    "31R": "프리미어리그 31라운드(vs 노팅엄 포레스트): 토트넘 0:3 패배. 토트넘이 득점하지 못한 경기.",
    "32R": "프리미어리그 32라운드(vs 선덜랜드): 토트넘 0:1 패배. 토트넘이 득점하지 못한 경기. 토트넘의 이전 감독 이고르 투도르의 마지막 경기이며, 이후 이고르 투도르 감독이 경질됨.",
    "33R": "프리미어리그 33라운드(vs 브라이튼): 토트넘 2:2 무승부. 토트넘과 상대팀 모두 득점한 경기. 토트넘의 현재 감독 로베르토 데 제르비가 부임한 후 첫 경기.",
    "34R": "프리미어리그 34라운드(vs 울버햄튼): 토트넘 1:0 승리. 상대팀은 득점하지 못한 경기.",
    "35R": "프리미어리그 35라운드(vs 아스톤 빌라): 토트넘 2:1 승리. 토트넘과 상대팀 모두 득점한 경기.",
    "36R": "프리미어리그 36라운드(vs 리즈 유나이티드): 토트넘 1:1 무승부. 토트넘과 상대팀 모두 득점한 경기.",
    "37R": "프리미어리그 37라운드(vs 첼시): 토트넘 1:2 패배. 토트넘과 상대팀 모두 득점한 경기.",
    "38R": "프리미어리그 38라운드(vs 에버튼): 토트넘 1:0 승리. 상대팀은 득점하지 못한 경기. 토트넘이 프리미어리그 잔류를 확정 지은 경기. 잔류 확정에 대한 안도, 기쁨, 조롱, 비꼼이 포함될 수 있음. 단, 웨스트햄 등 타 팀 강등에 대한 조롱은 토트넘 자체에 대한 감정이 아닐 수 있으므로 문맥에 따라 opponent 또는 neutral로 판단해야 함."
}

In [ ]:
# =====================================================
# 4. match_context 생성 함수
# =====================================================

def make_match_context(row):
    source = row.get("source")
    round_info = row.get("round")
    title = row.get("title")
    period = row.get("period")
    content_type = row.get("content_type")
    content_published_at = row.get("content_published_at")
    comment_published_at = row.get("comment_published_at")

    # 유튜브: 라운드별 경기 맥락 사용
    if source == "youtube":
        base_context = match_context_map.get(
            round_info,
            "토트넘 관련 유튜브 영상 댓글. 영상 제목과 댓글 내용을 바탕으로 판단."
        )

        return f"""
{base_context}

[데이터 정보]
데이터 출처: YouTube 댓글
영상 제목: {title}
라운드: {round_info}
영상 업로드일: {content_published_at}
댓글 작성일: {comment_published_at}
38R 기준 구분: {period}

이 댓글은 토트넘 관련 유튜브 영상에 달린 댓글이다.
영상 제목과 라운드 정보를 댓글 해석에 반영한다.
"""

    # 카페: 경기 하이라이트가 아니라 팬 커뮤니티 의견
    if source == "cafe":
        if period == "before_38R":
            return f"""
[데이터 정보]
데이터 출처: 토트넘 팬 카페
게시글 제목: {title}
데이터 유형: {content_type}
게시글 작성일: {content_published_at}
댓글 작성일: {comment_published_at}
38R 기준 구분: before_38R

이 텍스트는 38R 이전 토트넘 팬 카페 게시글 또는 댓글이다.
시즌 종료 직전 팬들의 기대, 우려, 경기 전 전망, 선수단 평가, 감독/전술 평가가 포함될 수 있다.
영상 장면에 대한 즉각 반응이 아니라 팬 커뮤니티의 의견으로 해석한다.
"""
        elif period == "after_38R":
            return f"""
[데이터 정보]
데이터 출처: 토트넘 팬 카페
게시글 제목: {title}
데이터 유형: {content_type}
게시글 작성일: {content_published_at}
댓글 작성일: {comment_published_at}
38R 기준 구분: after_38R

이 텍스트는 38R 이후 토트넘 팬 카페 게시글 또는 댓글이다.
시즌 종료 직후 팬들의 시즌 총평, 선수 평가, 감독 평가, 이적시장 의견, 구단 운영 평가가 포함될 수 있다.
영상 장면에 대한 즉각 반응이 아니라 팬 커뮤니티의 의견으로 해석한다.
"""
        else:
            return f"""
[데이터 정보]
데이터 출처: 토트넘 팬 카페
게시글 제목: {title}
데이터 유형: {content_type}

이 텍스트는 토트넘 팬 카페 게시글 또는 댓글이다.
팬 커뮤니티의 의견으로 해석한다.
"""

    return "토트넘 관련 텍스트 데이터."


df["match_context"] = df.apply(make_match_context, axis=1)

# 같은 맥락끼리 묶기 위한 key
df["context_key"] = (
    df["source"].astype(str) + " | " +
    df["round"].astype(str) + " | " +
    df["period"].astype(str) + " | " +
    df["title"].astype(str)
)

print("match_context 결측 수:", df["match_context"].isna().sum())

display(
    df[[
        "source",
        "round",
        "period",
        "title",
        "content_type",
        "text_clean",
        "match_context"
    ]].head()
)

In [ ]:
# =====================================================
# 5. 1차 감정분석 함수
# =====================================================

def analyze_batch(comments, match_context):
    comments_text = "\n".join([
        f"{i}. {comment}"
        for i, comment in enumerate(comments)
    ])

    prompt = f"""
너는 한국어 축구 팬덤 텍스트를 분석하는 target-aware 감정분석 분류기다.
분석 대상은 EPL 토트넘 관련 유튜브 댓글과 팬 카페 게시글/댓글이다.

[토트넘 선수단 정보]
현재 감독: 로베르토 데 제르비
이전 감독: 이고르 투도르

선수:
굴리엘모 비카리오, 라구 드라구신, 케빈 단소, 사비 시몬스, 히샬리송, 제임스 매디슨, 마티스 텔,
데스티니 우도기, 아치 그레이, 루카스 베리발, 크리스티안 로메로, 도미닉 솔랑케,
모하메드 쿠드스, 데얀 클루셉스키, 코너 갤러거, 페드로 포로, 제드 스펜스, 윌손 오도베르,
파페 마타르 사르, 로드리고 벤탕쿠르, 안토닌 킨스키, 벤 데이비스, 미키 판더펜,
소우자, 랑달 콜로 무아니, 브랜던 오스틴

[선수단 반영 규칙]
- 위 선수나 감독을 칭찬하면 토트넘 관련 positive로 분류한다.
- 위 선수나 감독을 비판, 조롱, 실망하면 토트넘 관련 negative로 분류한다.
- 위 선수 이름이 직접 나오면 target은 보통 tottenham_player로 둔다.
- 로베르토 데 제르비, 이고르 투도르 관련 댓글이면 target은 tottenham_manager로 둔다.
- 토트넘 이외의 다른 대상을 언급한 댓글은 토트넘 감정으로 보지 않고 neutral로 분류한다.

[데이터 맥락]
{match_context}

위 데이터 맥락은 반드시 반영하되, source에 따라 다르게 해석한다.

[데이터 출처별 해석 규칙]

1. source가 youtube인 경우
- 유튜브 댓글은 특정 영상 또는 경기 장면에 대한 즉각적인 반응일 수 있다.
- 영상 제목, 라운드, 경기 결과, 스코어 맥락을 적극 반영한다.
- 댓글이 짧거나 대상이 생략되어도 영상/경기 맥락상 토트넘에 대한 반응으로 자연스럽게 해석되면 positive 또는 negative로 판단한다.
- 토트넘이 승리한 경기에서는 "골 미쳤다", "마무리 좋다", "슈팅 지렸다", "수비 좋다"처럼 대상이 불명확한 경기력 칭찬은 특별히 상대팀 언급이 없으면 토트넘 positive로 우선 판단한다.
- 토트넘이 패배한 경기에서는 아쉬움, 비판, 조롱, 실망 표현이 토트넘을 향하면 negative로 판단한다.
- 단, 패배 경기라도 토트넘 선수 개인의 좋은 플레이를 칭찬하면 positive로 판단한다.

2. source가 cafe인 경우
- 카페 게시글/댓글은 특정 경기 장면에 대한 즉각 반응으로 단정하지 않는다.
- 경기 스코어보다 게시글 제목, 작성 시점, 38R 전후 맥락, 본문/댓글의 전체 의미를 우선한다.
- 카페 데이터는 시즌 총평, 선수 평가, 감독/전술 평가, 이적시장 의견, 구단 운영 평가, 팬덤 내 토론이 중심일 수 있다.
- 단순 이적 루머 전달, 정보 공유, 질문, 일정 확인은 neutral로 둔다.
- 토트넘 선수/감독/구단 운영/전술/시즌 결과에 대한 기대, 칭찬, 만족은 positive로 본다.
- 토트넘 선수/감독/구단 운영/전술/시즌 결과에 대한 실망, 비판, 조롱, 불안, 체념은 negative로 본다.
- 카페 댓글이 게시글 제목에 반응하는 형태라면, 게시글 제목의 맥락을 함께 고려한다.

[분류 기준]
- positive: 토트넘 팀/선수/감독/전술/경기력/결과/구단 운영에 대한 칭찬, 감탄, 기대, 축하, 만족
- negative: 토트넘 팀/선수/감독/전술/경기력/결과/구단 운영에 대한 비판, 실망, 조롱, 비꼼, 분노, 불안, 체념
- neutral: 토트넘과 직접 관련 없거나 단순 질문/정보/감정 판단 불가

[중요 규칙]
- neutral은 정말 감정 방향을 판단할 수 없을 때만 사용한다.
- 토트넘 관련 맥락이 있고 긍정/부정 뉘앙스가 조금이라도 있으면 positive 또는 negative로 판단한다.
- "미쳤다", "지린다", "개쩐다", "폼 미쳤다", "찢었다", "월클", "GOAT", "레전드", "나이스"는 칭찬 맥락이면 positive.
- "개못한다", "답답하다", "망했다", "처참하다", "노답", "왜 저러냐", "다 나가", "상장폐지", "강등권"은 비판/조롱 맥락이면 negative.
- 상대팀/상대선수 칭찬은 토트넘과 무관하면 neutral.
- 단, 상대팀 칭찬이 토트넘의 패배, 부진, 약점, 비교 열세를 강조하면 negative.
- 단순 감탄사라도 유튜브 영상 맥락상 토트넘 경기 장면에 대한 반응으로 해석 가능하면 neutral보다 positive/negative를 우선한다.
- 카페 데이터에서는 단순 감탄사보다 게시글 제목과 문장 전체 의미를 더 중요하게 본다.

[target 기준]
- tottenham_team
- tottenham_player
- tottenham_manager
- opponent
- referee_or_league
- broadcast
- unclear

[target 판단 규칙]
- 토트넘 팀 전체, 경기력, 순위, 시즌 결과, 구단 분위기 관련이면 tottenham_team.
- 토트넘 선수 개인 관련이면 tottenham_player.
- 감독, 전술, 교체, 라인업 관련이면 tottenham_manager.
- 상대팀이나 상대선수가 중심이면 opponent.
- 심판, 리그, 판정, 일정, 대회 구조가 중심이면 referee_or_league.
- 유튜브 방송, 출연진, 편집, 해설, 채널 자체에 대한 반응이면 broadcast.
- 대상이 불명확하면 unclear.

[출력 규칙]
반드시 JSON 배열만 출력하라.
설명 문장, 코드블록, 마크다운은 쓰지 마라.
각 결과에는 입력 번호 id를 포함하라.

[
  {{
    "id": 0,
    "sentiment": "positive 또는 negative 또는 neutral",
    "confidence": 0.0부터 1.0 사이 숫자,
    "target": "tottenham_team 또는 tottenham_player 또는 tottenham_manager 또는 opponent 또는 referee_or_league 또는 broadcast 또는 unclear",
    "reason": "짧은 한국어 이유"
  }}
]

[분석할 텍스트]
{comments_text}
"""

    try:
        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[
                {"role": "system", "content": "너는 한국어 축구 댓글 감정분석 전문가다."},
                {"role": "user", "content": prompt}
            ],
            temperature=0
        )

        result_text = response.choices[0].message.content.strip()
        return parse_json_array(result_text)

    except Exception as e:
        return [
            {
                "id": i,
                "sentiment": "error",
                "confidence": 0.0,
                "target": "error",
                "reason": str(e)
            }
            for i in range(len(comments))
        ]

In [ ]:
# =====================================================
# 6. 2차 neutral 재분석 함수
# =====================================================

def analyze_neutral_batch(comments, match_context):
    comments_text = "\n".join([
        f"{i}. {comment}"
        for i, comment in enumerate(comments)
    ])

    prompt = f"""
너는 토트넘 관련 유튜브 댓글과 팬 카페 게시글/댓글의 2차 감정분석 분류기다.
아래 텍스트들은 1차 분석에서 neutral로 분류되었지만, 실제로 긍정/부정 뉘앙스가 숨어 있는지 재검토한다.

[데이터 맥락]
{match_context}

[가장 중요한 원칙]
neutral은 감정 방향이 정말 없을 때만 유지한다.
하지만 모든 텍스트를 억지로 positive/negative로 바꾸지는 않는다.

칭찬, 감탄, 만족, 축하, 기대가 조금이라도 있으면 positive.
비판, 실망, 조롱, 분노, 답답함, 비꼼, 불안, 체념이 조금이라도 있으면 negative.
감정 단서가 없고 단순 정보/질문/사실 전달이면 neutral.

[데이터 출처별 재검토 기준]

1. source가 youtube인 경우
- 유튜브 댓글은 특정 영상/경기 장면에 대한 짧은 반응일 수 있다.
- 댓글이 짧거나 대상이 생략되어도 영상/경기 맥락상 토트넘에 대한 반응으로 자연스럽게 해석되면 positive 또는 negative로 분류한다.
- "미쳤다", "지렸다", "좋다", "깔끔하다", "나이스", "레전드", "GOAT"는 토트넘 장면/선수/경기력 칭찬이면 positive.
- "답답하다", "망했다", "왜 저러냐", "최악", "또 이러네", "아쉽다", "다 나가", "상장폐지", "강등권"은 토트넘 비판/조롱이면 negative.
- 방송 출연진, 편집, 해설, 유튜버 자체에 대한 반응이면 토트넘 감정이 아니므로 neutral 또는 broadcast 대상으로 둔다.

2. source가 cafe인 경우
- 카페 게시글/댓글은 특정 경기 장면 반응으로 단정하지 않는다.
- 게시글 제목, 작성 시점, 38R 전후 맥락, 문장 전체 의미를 종합해 판단한다.
- 시즌 총평, 선수 평가, 감독/전술 평가, 이적시장 의견, 구단 운영 평가에 대한 감정이 있으면 positive 또는 negative로 분류한다.
- 단순 이적 루머 전달, 정보 공유, 질문, 링크 공유, 일정 확인, 선수 이름만 언급한 글은 neutral로 유지한다.
- 기대감, 만족, 칭찬, 긍정적 전망은 positive.
- 불만, 조롱, 체념, 불안, 실망, 비판, 부정적 전망은 negative.
- 카페 댓글이 게시글 제목에 대한 짧은 반응이면 게시글 제목의 맥락을 함께 고려한다.

[neutral로 유지하는 경우]
- 단순 질문
- 경기 시간, 일정, 정보 확인
- 단순 이적 루머/기사 공유
- 선수 이름이나 팀 이름만 언급
- 영상 편집, 중계, 해설, 업로드 관련 이야기
- 토트넘 / 토트넘 선수 / 감독 / 구단 운영과 감정적으로 연결되지 않는 말
- 감정 단서가 전혀 없는 단순 사실 전달

[neutral에서 바꿔야 하는 경우]
- "역시 토트넘"처럼 비꼼/조롱으로 해석되면 negative.
- "이 정도면 괜찮다", "기대된다", "좋은 영입"처럼 긍정 평가가 있으면 positive.
- "답 없다", "불안하다", "또 망했다", "이러면 안 된다"처럼 부정 평가가 있으면 negative.
- 상대팀 칭찬이라도 토트넘의 패배, 약점, 부진, 비교 열세를 강조하면 negative.

[판단 우선순위]
1. 비판 / 조롱 / 실망 / 불안 / 체념 단서가 있으면 negative
2. 칭찬 / 감탄 / 만족 / 기대 단서가 있으면 positive
3. 감정 단서가 없고 정보성 텍스트이면 neutral

[target 기준]
- tottenham_team
- tottenham_player
- tottenham_manager
- opponent
- referee_or_league
- broadcast
- unclear

[출력 규칙]
반드시 JSON 배열만 출력한다.
설명 문장, 코드블록, 마크다운은 출력하지 않는다.
각 결과에는 입력 번호 id를 포함한다.

[
  {{
    "id": 0,
    "sentiment": "positive 또는 negative 또는 neutral",
    "confidence": 0.0,
    "target": "tottenham_team 또는 tottenham_player 또는 tottenham_manager 또는 opponent 또는 referee_or_league 또는 broadcast 또는 unclear",
    "reason": "짧은 한국어 이유"
  }}
]

[재분석할 텍스트]
{comments_text}
"""

    try:
        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[
                {
                    "role": "system",
                    "content": "너는 한국어 축구 댓글 감정분석 전문가다. neutral 판정은 매우 보수적으로 사용한다."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0
        )

        result_text = response.choices[0].message.content.strip()
        return parse_json_array(result_text)

    except Exception as e:
        return [
            {
                "id": i,
                "sentiment": "neutral",
                "confidence": 0.0,
                "target": "unclear",
                "reason": str(e)
            }
            for i in range(len(comments))
        ]

In [ ]:
# =====================================================
# 7. 1차 감정분석 실행
# =====================================================

def run_first_pass(df, batch_size=20):
    final_df = df.copy()

    final_df["new_gpt_sentiment"] = None
    final_df["new_gpt_confidence"] = None
    final_df["new_gpt_target"] = None
    final_df["new_gpt_reason"] = None

    for context_key, group_df in tqdm(final_df.groupby("context_key"), desc="1차 감정분석"):
        match_context = group_df["match_context"].iloc[0]

        group_df = group_df.reset_index()

        for start in range(0, len(group_df), batch_size):
            batch_df = group_df.iloc[start:start+batch_size]
            comments = batch_df["text_clean"].tolist()

            batch_results = analyze_batch(comments, match_context)

            result_map = {}

            for r in batch_results:
                try:
                    result_map[int(r.get("id"))] = r
                except:
                    pass

            for local_id, row in enumerate(batch_df.itertuples()):
                r = result_map.get(local_id, {})

                idx = row.index

                sentiment = str(r.get("sentiment", "error")).lower().strip()

                if sentiment not in ["positive", "negative", "neutral", "error"]:
                    sentiment = "error"

                final_df.at[idx, "new_gpt_sentiment"] = sentiment
                final_df.at[idx, "new_gpt_confidence"] = r.get("confidence", 0.0)
                final_df.at[idx, "new_gpt_target"] = r.get("target", "unclear")
                final_df.at[idx, "new_gpt_reason"] = r.get("reason", "")

            time.sleep(0.3)

    return final_df


labeled_df = run_first_pass(df, batch_size=20)

print("1차 감정분석 결과")
print(labeled_df["new_gpt_sentiment"].value_counts(dropna=False))

In [ ]:
# =====================================================
# 8. 2차 neutral 재분석
# =====================================================

def run_second_pass_for_neutral(labeled_df, batch_size=20):
    final_df = labeled_df.copy()

    final_df["gpt_sentiment_1st"] = final_df["new_gpt_sentiment"]
    final_df["gpt_reason_1st"] = final_df["new_gpt_reason"]
    final_df["neutral_second_pass"] = False

    neutral_df = final_df[
        final_df["new_gpt_sentiment"].astype(str).str.lower().eq("neutral")
    ].copy()

    print("2차 재분석 대상 neutral 수:", len(neutral_df))

    for context_key, group_df in tqdm(neutral_df.groupby("context_key"), desc="2차 neutral 재분석"):
        match_context = group_df["match_context"].iloc[0]

        group_df = group_df.reset_index()

        for start in range(0, len(group_df), batch_size):
            batch_df = group_df.iloc[start:start+batch_size]
            comments = batch_df["text_clean"].tolist()

            batch_results = analyze_neutral_batch(comments, match_context)

            result_map = {}

            for r in batch_results:
                try:
                    result_map[int(r.get("id"))] = r
                except:
                    pass

            for local_id, row in enumerate(batch_df.itertuples()):
                r = result_map.get(local_id, {})

                idx = row.index

                sentiment = str(r.get("sentiment", "neutral")).lower().strip()

                if sentiment not in ["positive", "negative", "neutral"]:
                    sentiment = "neutral"

                final_df.at[idx, "new_gpt_sentiment"] = sentiment
                final_df.at[idx, "new_gpt_confidence"] = r.get("confidence", 0.0)
                final_df.at[idx, "new_gpt_target"] = r.get("target", "unclear")
                final_df.at[idx, "new_gpt_reason"] = "[2차 neutral 재분석] " + str(r.get("reason", ""))
                final_df.at[idx, "neutral_second_pass"] = True

            time.sleep(0.3)

    return final_df


labeled_df = run_second_pass_for_neutral(labeled_df, batch_size=20)

print("2차 재분석 후 결과")
print(labeled_df["new_gpt_sentiment"].value_counts(dropna=False))

In [ ]:
# =====================================================
# 9. 기존 gpt 컬럼 갱신
# =====================================================

labeled_df["gpt_sentiment_old"] = labeled_df["gpt_sentiment"]
labeled_df["gpt_confidence_old"] = labeled_df["gpt_confidence"]
labeled_df["gpt_target_old"] = labeled_df["gpt_target"]
labeled_df["gpt_reason_old"] = labeled_df["gpt_reason"]

labeled_df["gpt_sentiment"] = labeled_df["new_gpt_sentiment"]
labeled_df["gpt_confidence"] = labeled_df["new_gpt_confidence"]
labeled_df["gpt_target"] = labeled_df["new_gpt_target"]
labeled_df["gpt_reason"] = labeled_df["new_gpt_reason"]


# =====================================================
# 10. 저장
# =====================================================

date_cols = [
    "content_published_at",
    "comment_published_at"
]

labeled_df_excel = labeled_df.copy()

for col in date_cols:
    if col in labeled_df_excel.columns:
        labeled_df_excel[col] = pd.to_datetime(
            labeled_df_excel[col],
            errors="coerce",
            utc=True
        ).dt.tz_convert(None)

labeled_df.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig"
)

labeled_df_excel.to_excel(
    OUTPUT_XLSX,
    index=False
)

print("✅ 저장 완료")
print("CSV:", OUTPUT_CSV)
print("Excel:", OUTPUT_XLSX)


# =====================================================
# 11. 결과 미리보기
# =====================================================

display(
    labeled_df[
        [
            "source",
            "round",
            "period",
            "title",
            "content_type",
            "text_clean",
            "gpt_sentiment_1st",
            "gpt_sentiment",
            "gpt_confidence",
            "gpt_target",
            "gpt_reason",
            "neutral_second_pass"
        ]
    ].head(20)
)